# Kirby EEG: frozen replication on subjects 2 and 3
Use a GPU runtime and enable the OPENAI_API_KEY Colab Secret. Run all cells and upload the updated kirby_eeg_colab.zip.
12 models: two new subjects, three seeds, clean/blink. Same run6 repair procedure; A and B both retained.
Maximum 144 API requests and $100 estimated token cost (not a billing guarantee). No calls until development gates pass.
Keep all exclusions and failures. Do not tune against final results. The plan and source hashes are checked before execution.
Rerunning cells reuses the saved replication ID; download the evidence ZIP when finished.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, os, stat, subprocess, sys, uuid, zipfile
from google.colab import files

WORKSPACE = Path('/content/kirby-eeg-workspace')
WORKSPACE.mkdir(exist_ok=True)
if 'EEG_SOURCE_ZIP' not in globals():
    print('Upload kirby_eeg_colab.zip')
    uploaded = files.upload()
    names = [name for name in uploaded if Path(name).name == 'kirby_eeg_colab.zip']
    if len(names) != 1:
        raise ValueError('Upload exactly the new kirby_eeg_colab.zip source bundle')
    EEG_SOURCE_ZIP = Path(names[0]).resolve()
source_hash = hashlib.sha256(EEG_SOURCE_ZIP.read_bytes()).hexdigest()
SOURCE = WORKSPACE / ('source-' + source_hash[:16])
SOURCE.mkdir(exist_ok=True)
with zipfile.ZipFile(EEG_SOURCE_ZIP) as archive:
    entries = archive.infolist()
    if len(entries) > 200 or sum(i.file_size for i in entries) > 10_000_000:
        raise ValueError('Unexpected source archive size')
    names = [i.filename for i in entries]
    if len(names) != len(set(names)):
        raise ValueError('Duplicate archive paths')
    for entry in entries:
        path = PurePosixPath(entry.filename)
        if (path.is_absolute() or '..' in path.parts or '\\' in entry.filename or ':' in entry.filename
                or not path.parts or path.parts[0] != 'backend'
                or stat.S_ISLNK(entry.external_attr >> 16)
                or (path.suffix not in ('.py', '.md', '.toml', '.ipynb')
                    and entry.filename != 'backend/experiments/eeg_replication_v1.json')):
            raise ValueError('Unsafe source archive member')
        target = SOURCE.joinpath(*path.parts)
        if not target.resolve().is_relative_to(SOURCE.resolve()):
            raise ValueError('Archive escaped source workspace')
    archive.extractall(SOURCE)
BACKEND = SOURCE / 'backend'
if not (BACKEND / 'kirby/eeg/runner.py').is_file():
    raise ValueError('This is not the EEG source bundle')
VENV = WORKSPACE / ('venv-' + source_hash[:16])
PYTHON = VENV / 'bin/python'
def worker_ready():
    if not PYTHON.exists():
        return False
    probe = subprocess.run([str(PYTHON), '-c',
        'import sys, pip; from pathlib import Path; assert sys.prefix != sys.base_prefix; '
        'assert Path(sys.prefix).resolve() == Path(sys.argv[1]).resolve()', str(VENV)],
        capture_output=True, text=True)
    return probe.returncode == 0

if not worker_ready():
    # Some Colab images omit ensurepip. Also repair partially created workers:
    # a python executable alone does not mean the environment is usable.
    subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(VENV)], check=True)
    if not worker_ready():
        # Base pip manages the target interpreter without requiring its ensurepip.
        subprocess.run([sys.executable, '-m', 'pip', '--python', str(PYTHON),
                        'install', '--disable-pip-version-check', 'pip'], check=True)
    if not worker_ready():
        raise RuntimeError('The EEG worker is not usable; inspect the setup output above before running an experiment.')
# Reuses Colab's CUDA torch; EEG dependencies live in the dedicated worker environment.
if not (VENV / 'kirby-installed.json').exists():
    subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check', '-e', str(BACKEND) + '[eeg]'], check=True)
    (VENV / 'kirby-installed.json').write_text(json.dumps({'source_sha256': source_hash}))
env = os.environ.copy()
env['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
env['MPLBACKEND'] = 'Agg'
env['MNE_DATA'] = str(WORKSPACE / 'dataset-cache')
Path(env['MNE_DATA']).mkdir(exist_ok=True)
subprocess.run([str(PYTHON), '-c', 'import torch; print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())'], env=env, check=True)
print('EEG workspace ready:', WORKSPACE)

In [ ]:
SUBJECTS = [2, 3]
SEEDS = [0, 1, 2]
CONDITIONS = ['clean', 'eye_blink']
EVIDENCE_CONDITIONS = ['A', 'B']
EPOCHS = 300
PATIENCE = 30
ATTEMPTS = 5
REPAIR_POLICY = 'feedback_guided'
RUN_ASTRA = True
EXTENDED_BASELINES = False
MAX_API_CALLS = 144
MAX_ESTIMATED_USD = 100.0
RUN_ID = None

from pathlib import Path
import hashlib, json
_plan_path = BACKEND / 'experiments/eeg_replication_v1.json'
_plan_sha256 = '826af2e339aea7c05a6de7dd951f9ca11506e195b20adb26b6749696d41b169c'
_replication_fields = {'SUBJECTS': 'subjects', 'SEEDS': 'seeds', 'CONDITIONS': 'conditions', 'EVIDENCE_CONDITIONS': 'evidence_conditions', 'EPOCHS': 'epochs', 'PATIENCE': 'patience', 'ATTEMPTS': 'attempts', 'REPAIR_POLICY': 'repair_policy', 'RUN_ASTRA': 'astra', 'EXTENDED_BASELINES': 'extended_baselines', 'MAX_API_CALLS': 'max_api_calls', 'MAX_ESTIMATED_USD': 'max_estimated_usd'}

def verify_replication():
    if hashlib.sha256(_plan_path.read_text(encoding='utf-8').encode('utf-8')).hexdigest() != _plan_sha256:
        raise ValueError('Replication plan changed. Rebuild and declare a new study before running.')
    plan = json.loads(_plan_path.read_text())
    for name, key in _replication_fields.items():
        if globals()[name] != plan['config'][key]:
            raise ValueError('Frozen replication setting changed: ' + name)
    for relative, expected in plan['source_sha256'].items():
        if hashlib.sha256((BACKEND / relative).read_text(encoding='utf-8').encode('utf-8')).hexdigest() != expected:
            raise ValueError('Frozen experiment source changed: ' + relative)
    print('Replication verified: subjects 2 and 3, seeds 0/1/2, unchanged run6 repair procedure.', flush=True)

verify_replication()

In [ ]:
verify_replication()

settings = {name: globals()[name] for name in (
    'SUBJECTS', 'SEEDS', 'CONDITIONS', 'EVIDENCE_CONDITIONS', 'EPOCHS', 'PATIENCE',
    'ATTEMPTS', 'REPAIR_POLICY', 'RUN_ASTRA', 'EXTENDED_BASELINES', 'MAX_API_CALLS', 'MAX_ESTIMATED_USD')}
settings['source_sha256'] = source_hash
attempt_file = WORKSPACE / 'eeg-replication-v1-attempt-id.txt'
if RUN_ID is not None:
    run_id = str(uuid.UUID(RUN_ID))
elif attempt_file.exists():
    run_id = str(uuid.UUID(attempt_file.read_text().strip()))
else:
    run_id = str(uuid.uuid4())
attempt_file.write_text(run_id)
run_root = WORKSPACE / 'eeg-runs' / run_id
settings_file = WORKSPACE / ('settings-' + run_id + '.json')
if settings_file.exists() and json.loads(settings_file.read_text()) != settings:
    raise ValueError('This run ID belongs to different settings/source. Use a deliberate new UUID for a new experiment.')
settings_file.write_text(json.dumps(settings, sort_keys=True))
if run_root.exists():
    print('Run already exists: no training, API calls, or final evaluation will be repeated.')
    if (run_root / 'report.json').exists():
        print('Saved status:', json.loads((run_root / 'report.json').read_text())['status'])
else:
    if RUN_ASTRA:
        from google.colab import userdata
        key = userdata.get('OPENAI_API_KEY')
        if not key:
            raise ValueError('Add OPENAI_API_KEY to Colab Secrets and enable notebook access.')
        env['OPENAI_API_KEY'] = key
        del key
    command = [str(PYTHON), '-u', '-m', 'kirby.eeg', 'run', '--output', str(WORKSPACE / 'eeg-runs'),
               '--data-root', str(WORKSPACE / 'eeg-data'), '--run-id', run_id,
               '--subjects', *map(str, SUBJECTS), '--seeds', *map(str, SEEDS),
               '--conditions', *CONDITIONS, '--evidence-conditions', *EVIDENCE_CONDITIONS,
               '--epochs', str(EPOCHS), '--patience', str(PATIENCE), '--attempts', str(ATTEMPTS),
               '--repair-policy', REPAIR_POLICY,
               '--max-api-calls', str(MAX_API_CALLS), '--max-estimated-usd', str(MAX_ESTIMATED_USD)]
    if not RUN_ASTRA:
        command.append('--no-astra')
    if EXTENDED_BASELINES:
        command.append('--extended-baselines')
    try:
        print('Starting experiment:', run_id, '| repair policy:', REPAIR_POLICY, flush=True)
        with subprocess.Popen(command, cwd=BACKEND, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as worker:
            try:
                for line in worker.stdout:
                    print(line, end='', flush=True)
                returncode = worker.wait()
            except BaseException:
                worker.terminate()
                try:
                    worker.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    worker.kill()
                    worker.wait()
                raise
        print('Worker exit code:', returncode, flush=True)
    finally:
        env.pop('OPENAI_API_KEY', None)
if (run_root / 'START_HERE.md').exists():
    from IPython.display import Markdown, display
    display(Markdown((run_root / 'START_HERE.md').read_text()))

if run_root.exists():
    (run_root / 'replication-plan.json').write_bytes(_plan_path.read_bytes())

In [ ]:
# Re-export reads saved results only. It never resumes a failed experiment.
if not (run_root / 'report.json').exists():
    raise RuntimeError('No report was created. Inspect the worker error above.')
subprocess.run([str(PYTHON), '-m', 'kirby.eeg', 'export', str(run_root)], cwd=BACKEND, env=env, check=True)
evidence = run_root / ('kirby-eeg-evidence-' + run_id + '.zip')
files.download(str(evidence))
print('Download includes current-run status, evidence, checkpoints and plots. Preserve it before resetting Colab.')